# Berlin BnB Lab: a small machine learning web app

Everything in one notebook:

1. **ML models**: linear regression (predict the price) and logistic regression (predict the room type), saved with joblib
2. **Backend**: FastAPI with `POST /predict/regression` and `POST /predict/classification`
3. **Frontend**: one HTML page with a form for each model

Keep `airbnb_berlin_synthetic_10000.csv` in the same folder as this notebook, then run the cells from top to bottom.

In [20]:
# Install the libraries (you can skip this cell if you already have them)
%pip install fastapi uvicorn scikit-learn pandas joblib requests

## Part 1: Machine learning models

In [21]:
import json
import threading
import time
from pathlib import Path

import joblib
import pandas as pd
import requests
import uvicorn
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Settings
DATA_FILE = Path(r"/content/airbnb_berlin_synthetic_10000.csv")
REG_FILE = Path("price_model.joblib")
CLF_FILE = Path("room_type_model.joblib")
METRICS_FILE = Path("metrics.json")

# Regression: predict the nightly price from these columns
REG_TEXT_COLS = ["neighbourhood", "room_type"]
REG_NUM_COLS = ["accommodates", "bedrooms", "bathrooms", "minimum_nights"]
REG_COLS = REG_TEXT_COLS + REG_NUM_COLS

# Classification: predict the room type from these columns
CLF_COLS = ["price", "accommodates", "bedrooms", "beds", "bathrooms", "minimum_nights"]

NEIGHBOURHOODS = [
    "Charlottenburg", "Friedrichshain", "Kreuzberg", "Lichtenberg", "Mitte",
    "Neukoelln", "Pankow", "Prenzlauer Berg", "Spandau", "Tempelhof",
]
ROOM_TYPES = ["Entire home/apt", "Hotel room", "Private room", "Shared room"]

In [22]:
# Load the data and take a look
df = pd.read_csv(DATA_FILE)
print(df.shape)
df.head()

(10000, 20)


,listing_id,neighbourhood,room_type,accommodates,bedrooms,beds,bathrooms,minimum_nights,availability_365,instant_bookable,host_total_listings,host_years,host_response_rate,host_acceptance_rate,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_communication,price,host_is_superhost
0,1,Mitte,Private room,3,1,2,1.0,3,301,0,2,0.7,95,70,18,4.76,4.74,4.62,83.0,0
1,2,Tempelhof,Entire home/apt,3,1,2,1.5,1,344,0,1,8.5,90,83,17,4.45,4.60,4.43,124.0,0
2,3,Neukoelln,Private room,1,1,1,1.0,1,260,0,2,1.8,97,77,10,5.00,5.00,5.00,31.0,1
3,4,Tempelhof,Private room,3,1,2,0.5,1,95,0,1,2.0,87,93,13,4.24,3.94,4.22,44.0,1
4,5,Friedrichshain,Entire home/apt,2,1,2,1.5,14,1,0,1,2.7,97,93,9,4.89,4.83,4.77,146.0,0


In [23]:
# Linear regression: predict the price (a number)
X = df[REG_COLS]
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Text columns become 0/1 columns; number columns are passed through
prepare = ColumnTransformer(
    [("text", OneHotEncoder(handle_unknown="ignore"), REG_TEXT_COLS)],
    remainder="passthrough",
)
reg_model = make_pipeline(prepare, LinearRegression())
reg_model.fit(X_train, y_train)

r2 = r2_score(y_test, reg_model.predict(X_test))
print(f"Regression R2: {r2:.3f}")

Regression R2: 0.797


In [24]:
# Classification: predict the room type (a category)
X = df[CLF_COLS]
y = df["room_type"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, clf_model.predict(X_test))
print(f"Classification accuracy: {accuracy:.3f}")

Classification accuracy: 0.867


In [25]:
# Save both models with joblib (and the scores, so the web page can show them)
joblib.dump(reg_model, REG_FILE)
joblib.dump(clf_model, CLF_FILE)
METRICS_FILE.write_text(json.dumps({"r2": round(r2, 3), "accuracy": round(accuracy, 3)}))
print("Saved:", REG_FILE, CLF_FILE, METRICS_FILE)

Saved: price_model.joblib room_type_model.joblib metrics.json


## Part 2: Backend (FastAPI)

In [26]:
# Load the saved models, exactly like a real backend would
reg_model = joblib.load(REG_FILE)
clf_model = joblib.load(CLF_FILE)

app = FastAPI(title="Berlin BnB Lab")


class PriceInput(BaseModel):
    neighbourhood: str
    room_type: str
    accommodates: int
    bedrooms: int
    bathrooms: float
    minimum_nights: int


class RoomTypeInput(BaseModel):
    price: float
    accommodates: int
    bedrooms: int
    beds: int
    bathrooms: float
    minimum_nights: int


@app.post("/predict/regression")
def predict_regression(data: PriceInput):
    row = pd.DataFrame([data.model_dump()])[REG_COLS]
    price = float(reg_model.predict(row)[0])
    return {"predicted_price": round(max(price, 0), 2)}


@app.post("/predict/classification")
def predict_classification(data: RoomTypeInput):
    row = pd.DataFrame([data.model_dump()])[CLF_COLS]
    label = str(clf_model.predict(row)[0])
    chances = clf_model.predict_proba(row)[0]
    probabilities = {str(name): round(float(p), 3) for name, p in zip(clf_model.classes_, chances)}
    return {"predicted_room_type": label, "probabilities": probabilities}


@app.get("/metrics")
def get_metrics():
    return json.loads(METRICS_FILE.read_text())

## Part 3: Frontend (one HTML page)

In [29]:
PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Berlin BnB Lab</title>
<style>
  :root {
    --bg: #14161a; --panel: #1d2026; --line: #343942; --text: #f2f2ee;
    --muted: #9aa0aa; --yellow: #ffd900; --red: #e3342f; --blue: #2b6cb0;
  }
  * { box-sizing: border-box; }
  body {
    margin: 0; padding: 32px 16px 48px; background: var(--bg); color: var(--text);
    font-family: "Helvetica Neue", Arial, sans-serif;
  }
  .wrap { max-width: 980px; margin: 0 auto; }

  /* station-sign header */
  header { display: flex; align-items: center; gap: 16px; margin-bottom: 28px; }
  .logo {
    background: var(--yellow); color: #000; font-weight: 900; font-size: 30px;
    width: 52px; height: 52px; display: grid; place-items: center; border-radius: 6px;
  }
  h1 { margin: 0; font-size: 28px; letter-spacing: 1px; text-transform: uppercase; }
  .sub { color: var(--muted); font-size: 14px; margin-top: 2px; }

  .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(380px, 1fr)); gap: 24px; }
  @media (max-width: 480px) { .grid { grid-template-columns: 1fr; } }

  /* each model is a "train line" card */
  .card { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; overflow: hidden; }
  .card-top { display: flex; align-items: center; gap: 12px; padding: 16px 20px; }
  .reg .card-top { background: var(--red); }
  .clf .card-top { background: var(--blue); }
  .badge {
    background: #fff; color: #000; font-weight: 900; border-radius: 50%;
    width: 36px; height: 36px; display: grid; place-items: center; font-size: 14px;
  }
  .card-top h2 { margin: 0; font-size: 18px; }
  .card-top small { opacity: .85; }
  form { padding: 20px; display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
  label { font-size: 12px; text-transform: uppercase; letter-spacing: .8px; color: var(--muted); display: block; }
  input, select {
    width: 100%; margin-top: 6px; padding: 10px; font-size: 15px; color: var(--text);
    background: var(--bg); border: 1px solid var(--line); border-radius: 6px;
  }
  input:focus, select:focus { outline: 2px solid var(--yellow); border-color: transparent; }
  .full { grid-column: 1 / -1; }
  button {
    grid-column: 1 / -1; padding: 12px; font-size: 15px; font-weight: 800; cursor: pointer;
    background: var(--yellow); color: #000; border: 0; border-radius: 6px; text-transform: uppercase;
    letter-spacing: 1px;
  }
  button:hover { filter: brightness(1.08); }

  /* result "departure board" */
  .result {
    margin: 0 20px 20px; padding: 16px; border-radius: 6px; background: #000;
    font-family: "Courier New", monospace; color: var(--yellow); min-height: 70px;
  }
  .result .big { font-size: 30px; font-weight: bold; }
  .result .hint { color: #6b7078; }
  .result .error { color: #ff7b72; }
  .bar-row { display: flex; align-items: center; gap: 8px; margin-top: 8px; font-size: 13px; color: #ddd; }
  .bar-row span:first-child { width: 120px; }
  .bar { flex: 1; height: 8px; background: #2a2d33; border-radius: 4px; overflow: hidden; }
  .bar div { height: 100%; background: var(--yellow); }

  footer { margin-top: 28px; color: var(--muted); font-size: 13px; text-align: center; }
</style>
</head>
<body>
<div class="wrap">

  <header>
    <div class="logo">U</div>
    <div>
      <h1>Berlin BnB Lab</h1>
      <div class="sub">Two tiny machine learning models, trained on 10,000 Berlin listings</div>
    </div>
  </header>

  <div class="grid">

    <!-- Form 1: regression -->
    <section class="card reg">
      <div class="card-top">
        <div class="badge">U2</div>
        <div><h2>Price guesser</h2><small>Linear regression &middot; predicts a number</small></div>
      </div>
      <form id="regForm">
        <div><label>Neighbourhood<select name="neighbourhood" id="regHood"></select></label></div>
        <div><label>Room type<select name="room_type" id="regRoom"></select></label></div>
        <div><label>Guests<input type="number" name="accommodates" value="2" min="1" max="10"></label></div>
        <div><label>Bedrooms<input type="number" name="bedrooms" value="1" min="1" max="10"></label></div>
        <div><label>Bathrooms<input type="number" name="bathrooms" value="1" min="0.5" max="5" step="0.5"></label></div>
        <div><label>Minimum nights<input type="number" name="minimum_nights" value="2" min="1" max="365"></label></div>
        <button type="submit">Guess the price</button>
      </form>
      <div class="result" id="regResult"><span class="hint">Fill in the form to see a price...</span></div>
    </section>

    <!-- Form 2: classification -->
    <section class="card clf">
      <div class="card-top">
        <div class="badge">U8</div>
        <div><h2>Room type detective</h2><small>Logistic regression &middot; predicts a category</small></div>
      </div>
      <form id="clfForm">
        <div><label>Price per night (EUR)<input type="number" name="price" value="90" min="1" step="1"></label></div>
        <div><label>Guests<input type="number" name="accommodates" value="2" min="1" max="10"></label></div>
        <div><label>Bedrooms<input type="number" name="bedrooms" value="1" min="1" max="10"></label></div>
        <div><label>Beds<input type="number" name="beds" value="1" min="1" max="10"></label></div>
        <div><label>Bathrooms<input type="number" name="bathrooms" value="1" min="0.5" max="5" step="0.5"></label></div>
        <div><label>Minimum nights<input type="number" name="minimum_nights" value="2" min="1" max="365"></label></div>
        <button type="submit">Find the room type</button>
      </form>
      <div class="result" id="clfResult"><span class="hint">Fill in the form to see a room type...</span></div>
    </section>

  </div>

  <footer id="metrics"></footer>
</div>

<script>
  const NEIGHBOURHOODS = __NEIGHBOURHOODS__;
  const ROOM_TYPES = __ROOM_TYPES__;

  // fill the dropdowns
  function fillSelect(id, items) {
    document.getElementById(id).innerHTML = items.map(x => "<option>" + x + "</option>").join("");
  }
  fillSelect("regHood", NEIGHBOURHOODS);
  fillSelect("regRoom", ROOM_TYPES);

  // read a form into a plain object (number inputs become numbers)
  function readForm(form) {
    const data = {};
    for (const el of form.elements) {
      if (!el.name) continue;
      data[el.name] = el.type === "number" ? Number(el.value) : el.value;
    }
    return data;
  }

  // send the form to the backend, then let "show" draw the answer
  async function send(url, form, box, show) {
    box.innerHTML = '<span class="hint">Thinking...</span>';
    try {
      const response = await fetch(url, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(readForm(form)),
      });
      if (!response.ok) throw new Error("Server said " + response.status);
      box.innerHTML = show(await response.json());
    } catch (err) {
      box.innerHTML = '<span class="error">Something went wrong: ' + err.message + "</span>";
    }
  }

  document.getElementById("regForm").addEventListener("submit", (e) => {
    e.preventDefault();
    send("/predict/regression", e.target, document.getElementById("regResult"), (r) =>
      '<div class="hint">Estimated price</div><div class="big">&euro; ' + r.predicted_price.toFixed(2) + " / night</div>"
    );
  });

  document.getElementById("clfForm").addEventListener("submit", (e) => {
    e.preventDefault();
    send("/predict/classification", e.target, document.getElementById("clfResult"), (r) => {
      let html = '<div class="hint">This looks like a</div><div class="big">' + r.predicted_room_type + "</div>";
      for (const [name, p] of Object.entries(r.probabilities)) {
        html += '<div class="bar-row"><span>' + name + '</span><div class="bar"><div style="width:' +
                (p * 100) + '%"></div></div><span>' + (p * 100).toFixed(0) + "%</span></div>";
      }
      return html;
    });
  });

  // show the model scores at the bottom
  fetch("/metrics").then(r => r.json()).then(m => {
    document.getElementById("metrics").textContent =
      "Model scores on unseen test data:  R\\u00b2 = " + m.r2 + "   |   accuracy = " + (m.accuracy * 100).toFixed(1) + "%";
  });
</script>
</body>
</html>
"""


@app.get("/", response_class=HTMLResponse)
def home():
    html = PAGE.replace("__NEIGHBOURHOODS__", json.dumps(NEIGHBOURHOODS))
    html = html.replace("__ROOM_TYPES__", json.dumps(ROOM_TYPES))
    return html

In [30]:
# ---- Run the frontend inside Google Colab ----
import os, time, threading, requests, uvicorn
from google.colab import output

# 1) Make the page use relative URLs, so it works behind Colab's proxy
PAGE = PAGE.replace('"/predict/', '"predict/').replace('fetch("/metrics")', 'fetch("metrics")')

# 2) Start the server on a free port (skips if it is already running)
PORT = 8000
try:
    server.should_exit = True      # stop an older server if you re-run this cell
    time.sleep(1)
except NameError:
    pass

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()

# 3) Wait until the server answers
for _ in range(30):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/metrics", timeout=1).ok:
            break
    except Exception:
        time.sleep(0.5)
print("Server is up")

# 4) Show the frontend right here in the notebook (both forms)
output.serve_kernel_port_as_iframe(PORT, height=900)

# Optional: also get a link to open it in its own browser tab
# output.serve_kernel_port_as_window(PORT)

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


Server is up


<IPython.core.display.Javascript object>

In [32]:
import threading, time, uvicorn
from google.colab import output

PAGE = PAGE.replace('"/predict/', '"predict/').replace('fetch("/metrics")', 'fetch("metrics")')

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
time.sleep(3)

output.serve_kernel_port_as_window(8000)

INFO:     Started server process [925]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>